# Imports & Setup

In [1]:
# 1. Install necessary library
!pip install -qU timm

# 2. Imports
import os
import gc
import math
import random
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 34.0 MB/s eta 0:00:0000:0100:01


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

# Configuration & Seeding

In [2]:
class Config:
    seed = 42
    # EVA-02 Large: A powerful transformer for fine-grained recognition
    model_name = "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k"
    
    img_size = 448
    
    num_classes = 31  # Adjust based on training set unique IDs if needed
    
    # Training Hyperparameters
    num_epochs = 1          # Increased slightly for better convergence
    batch_size = 4           # Keep small for large resolution
    grad_accum = 4           # Effective batch size = 16
    
    # Learning Rates (LLRD specific)
    encoder_lr = 2e-5        # Slower for the backbone
    head_lr = 1e-3           # Faster for the classifier head
    weight_decay = 1e-3      # Standard for ViT
    
    # ArcFace Hyperparameters
    arcface_s = 30.0
    arcface_m = 0.50
    
    # Advanced Options
    train_full_data = True  # Set TRUE for final submission, FALSE for validation
    n_folds = 5
    target_fold = 0          # Which fold to train on if validation is active
    
    use_tta = True           # Test Time Augmentation
    use_dba = True           # Test time DataBase augmentation
    use_qe = True            # Query Expansion
    use_rerank = True        # K-Reciprocal Re-ranking

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(Config.seed)

# Dataset & Advanced Transforms

In [3]:
# Stronger augmentations for the training set
train_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.RandomResizedCrop(size = (Config.img_size, Config.img_size), scale=(0.5, 0.8)),
    transforms.RandomHorizontalFlip(p=0.5),
    # TrivialAugmentWide: State-of-the-art auto-augmentation for small datasets
    transforms.TrivialAugmentWide(interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25),
])


# Clean transform for validation/testing
test_transform = transforms.Compose([
    transforms.Resize((Config.img_size, Config.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
class JaguarDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row["filename"]
        img_path = self.img_dir / img_name
        
        try:
            # Using RGB is important for EVA-02 which expects 3 channels
            img = Image.open(img_path).convert("RGB")
        except Exception:
            # Handle missing or corrupted files by returning a blank image
            img = Image.new("RGB", (Config.img_size, Config.img_size))

        if self.transform:
            img = self.transform(img)
            
        if self.is_test:
            return img, img_name
        
        # Return the pre-calculated integer index from our external mapping
        return img, torch.tensor(row["label_idx"], dtype=torch.long)

# Model (EVA-02 + ArcFace + Trainable GeM)

In [5]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        # p is now a learnable parameter
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)

class BalancedArcFace(nn.Module):
    def __init__(self, in_features, out_features, sample_counts, s_base=30.0, m=0.5):
        super().__init__()
        self.s_base = s_base
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        
        # Calculate class-specific scaling factors
        # Logic: s_i = s_base * (total_samples / (n_classes * count_i))
        # We use a log or power transform to prevent extreme scaling
        counts = torch.tensor(sample_counts, dtype=torch.float32)
        avg_count = counts.mean()
        
        # Adaptive factor: rare classes get > 1.0, common get < 1.0
        # We use a square root to dampen the effect so it doesn't explode
        self.register_buffer('s_factors', torch.sqrt(avg_count / counts))
        
    def forward(self, input, label=None):
        # 1. Standard Cosine Similarity
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        if label is None:
            return cosine
        
        # 2. Apply ArcFace Margin
        # Clamp for numerical stability (prevents NaNs)
        theta = torch.acos(cosine.clamp(-1.0 + 1e-7, 1.0 - 1e-7))
        phi = torch.cos(theta + self.m)
        
        # 3. Create One-Hot Mask
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1)
        
        # 4. Target class gets phi, others get cosine
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        
        # 5. Apply Class-Dependent Scaling
        # We fetch the specific s_factor for each image in the batch
        batch_s = self.s_base * self.s_factors[label] # Shape: (batch_size,)
        
        # Multiply each row by its specific scale factor
        output = logits * batch_s.view(-1, 1)
        
        return output

class EVABoss(nn.Module):
    def __init__(self, num_classes, class_counts):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.gem = GeM()
        self.bn = nn.BatchNorm1d(self.feat_dim)

        # USE BALANCED ARCFACE HERE
        self.head = BalancedArcFace(
            in_features= self.feat_dim, 
            out_features=num_classes, 
            sample_counts=class_counts, # Pass the list of counts per class
            s_base=Config.arcface_s, 
            m=Config.arcface_m
        )

    def forward(self, x, label=None):
        features = self.backbone.forward_features(x)
        
        # Standard reshaping logic for ViT/EVA
        if features.dim() == 3:
            B, N, C = features.shape
            H = W = int(math.sqrt(N))
            if H * W != N: features = features[:, -H*W:, :]
            features = features.permute(0, 2, 1).reshape(B, C, H, W)

        emb = self.gem(features).flatten(1)
        emb = self.bn(emb)
        
        # During training, 'label' is passed to BalancedArcFace to apply the margin
        # During inference (label=None), it returns simple cosine similarities
        return self.head(emb, label)

In [6]:
class DynamicSubCenterArcFace(nn.Module):
    def __init__(self, in_features, out_features, class_counts, s=30.0, m=0.50, max_k=3):
        super().__init__()
        self.s = s
        self.m = m
        self.max_k = max_k
        self.out_features = out_features

        # Allocate max_k centers for EVERY class
        self.weight = nn.Parameter(torch.FloatTensor(out_features * max_k, in_features))
        nn.init.xavier_uniform_(self.weight)

        # Calculate which classes get k=3 vs k=1
        counts = torch.tensor(class_counts, dtype=torch.float32)
        mean_count = counts.mean()

        # Create a boolean mask of shape (1, out_features, max_k)
        mask = torch.zeros(1, out_features, max_k, dtype=torch.bool)
        for i, count in enumerate(counts):
            if count > mean_count:
                mask[0, i, :] = True # Enable all 3 centers
            else:
                mask[0, i, 0] = True # Enable only 1 center (disable the other 2)
        
        # Register as a buffer so it moves to GPU automatically but doesn't require gradients
        self.register_buffer('mask', mask)

    def forward(self, input, label=None):
        # 1. Linear layer for cosine similarity
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        
        # 2. Reshape to group the sub-centers: (Batch, Classes, Max_K)
        cosine = cosine.view(-1, self.out_features, self.max_k)
        
        # 3. MASKING: Set disabled sub-centers to negative infinity
        # This ensures they are completely ignored by the max() function
        cosine = cosine.masked_fill(~self.mask, float('-inf'))
        
        # 4. Get the max similarity among the ACTIVE sub-centers
        cosine, _ = torch.max(cosine, dim=2)

        if label is None:
            return cosine

        # 5. Apply the ArcFace margin
        theta = torch.acos(cosine.clamp(-1.0 + 1e-6, 1.0 - 1e-6))
        phi = torch.cos(theta + self.m)
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, label.view(-1, 1), 1.0)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s


class EVABossDualHead(nn.Module):
    def __init__(self, num_classes, class_counts):
        super().__init__()
        self.backbone = timm.create_model(Config.model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.gem = GeM(p=3.0)
        self.bn = nn.BatchNorm1d(self.feat_dim)

        # HEAD 1: Phase 1 (Balanced)
        self.head_balanced = BalancedArcFace(
            in_features=self.feat_dim, 
            out_features=num_classes, 
            sample_counts=class_counts, 
            s_base=Config.arcface_s, 
            m=Config.arcface_m
        )

        # HEAD 2: Phase 2 (Sub-Center)
        self.head_subcenter = DynamicSubCenterArcFace(
            in_features=self.feat_dim, 
            out_features=num_classes, 
            class_counts=class_counts,
            s=Config.arcface_s, 
            m=Config.arcface_m,
            max_k=3
        )
        
        # Training Phase Toggle
        self.use_subcenter = False 

    def forward(self, x, label=None):
        features = self.backbone.forward_features(x)
        
        # ViT logic
        if features.dim() == 3:
            B, N, C = features.shape
            H = W = int(math.sqrt(N))
            if H * W != N: features = features[:, -H*W:, :]
            features = features.permute(0, 2, 1).reshape(B, C, H, W)

        emb = self.gem(features).flatten(1)
        emb = self.bn(emb)
        
        # Inference Mode
        if label is None:
            # Usually best to infer using the head you finished training with
            return self.head_subcenter(emb) if self.use_subcenter else self.head_balanced(emb)

        # Training Mode: Route to the active head
        if self.use_subcenter:
            return self.head_subcenter(emb, label)
        else:
            return self.head_balanced(emb, label)

In [7]:
def collaborative_loss(all_logits, labels, temperature=2.0, beta=0.5):
    """
    all_logits: List of logits from each head
    labels: Ground truth labels
    beta: Weight for the collaborative (agreement) part
    """
    num_heads = len(all_logits)
    total_loss = 0
    
    # 1. Standard Loss for each head (ArcFace)
    # This ensures every head learns the actual classes
    hard_loss = sum([F.cross_entropy(logits, labels) for logits in all_logits]) / num_heads
    
    # 2. Consensus Loss (The Distillation Part)
    # We want each head to match the 'average' prediction of the other heads
    soft_loss = 0
    with torch.no_grad():
        # Calculate the "Teacher" distribution (average of all heads)
        avg_probs = torch.stack([F.softmax(l / temperature, dim=1) for l in all_logits]).mean(dim=0)
        
    for logits in all_logits:
        # KL Divergence forces individual head to match the consensus
        p_i = F.log_softmax(logits / temperature, dim=1)
        soft_loss += F.kl_div(p_i, avg_probs, reduction='batchmean') * (temperature**2)
    
    soft_loss /= num_heads
    
    return hard_loss + (beta * soft_loss)

In [8]:
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        assert rho >= 0.0, f"Invalid rho, should be non-negative: {rho}"
        defaults = dict(rho=rho, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)  # climb to the peak
                self.state[p]["e_w"] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.sub_(self.state[p]["e_w"])  # get back to original weights
        self.base_optimizer.step()  # actual update step
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(torch.stack([p.grad.norm(p=2).to(shared_device) 
               for group in self.param_groups for p in group["params"] if p.grad is not None]), p=2)
        return norm

In [9]:
# Layer-wise Learning Rate Decay Optimizer
def get_optimizer_params(model, encoder_lr, head_lr, weight_decay=0.0):
    param_optimizer = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_parameters = []
    
    # Simple LLRD implementation
    layer_decay = 0.9
    num_layers = 24 # Approximate for Large models, or calculate dynamically
    
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        
        lr = encoder_lr
        if "head" in name:
            lr = head_lr
        elif "blocks" in name:
            try:
                # Attempt to extract block index to scale LR
                layer_id = int(name.split("blocks.")[1].split(".")[0])
                lr = encoder_lr * (layer_decay ** (num_layers - layer_id))
            except:
                pass
                
        if any(nd in name for nd in no_decay):
            optimizer_parameters.append({"params": [p], "weight_decay": 0.0, "lr": lr})
        else:
            optimizer_parameters.append({"params": [p], "weight_decay": weight_decay, "lr": lr})
            
    return optimizer_parameters

# Post-Processing Utils (QE & Re-ranking) 
## Added new reranking

In [10]:
def apply_dba(embeddings, k=3):
    # embeddings: (N, D)
    dist = torch.mm(embeddings, embeddings.t())
    _, indices = dist.topk(k, dim=1)
    
    # Average the features of the top-k neighbors
    dba_embeddings = embeddings[indices].mean(dim=1)
    # Re-normalize
    return F.normalize(dba_embeddings, p=2, dim=1)
    
import numpy as np
import torch

def query_expansion(emb, top_k=3):
    """
    Expands the query by averaging it with its top_k nearest neighbors.
    Uses alpha-weighting (cubed weights) for better rare-class retrieval.
    """
    # Safety Check: Convert PyTorch GPU tensor to NumPy
    if torch.is_tensor(emb):
        emb = emb.detach().cpu().numpy()
        
    print(f"Applying Weighted Query Expansion (k={top_k})...")
    
    # Cosine similarity matrix
    sims = emb @ emb.T
    
    # Get top k indices (descending similarity)
    # np.argsort(-sims) ensures we get the highest similarities first
    indices = np.argsort(-sims, axis=1)[:, :top_k]
    
    new_emb = np.zeros_like(emb)
    for i in range(len(emb)):
        # Similarity scores for the top_k neighbors
        neighbor_sims = sims[i, indices[i]]
        
        # Weighted average logic: Cubing weights emphasizes the closest neighbors
        # This helps 'Bernard' (13 images) by ignoring dissimilar neighbors
        weights = neighbor_sims ** 3  
        
        # Ensure weights sum to 1 to avoid feature scaling issues
        weights_sum = np.sum(weights)
        if weights_sum > 0:
            new_emb[i] = np.sum(emb[indices[i]] * weights[:, np.newaxis], axis=0) / weights_sum
        else:
            new_emb[i] = emb[i] # Fallback if something goes wrong

    # Re-normalize to unit sphere
    norm = np.linalg.norm(new_emb, axis=1, keepdims=True)
    return new_emb / (norm + 1e-12)

def k_reciprocal_rerank(prob, k1=20, k2=6, lambda_value=0.3):
    """
    Re-ranking using k-reciprocal encoding.
    Input 'prob' can be a NumPy array or a PyTorch GPU tensor.
    """
    # Safety Check: Convert to NumPy if it's a Tensor
    if torch.is_tensor(prob):
        prob = prob.detach().cpu().numpy()
        
    print("Applying K-Reciprocal Re-ranking...")
    
    # Distance matrix (Cosine distance = 1 - Cosine similarity)
    q_g_dist = 1 - prob
    original_dist = q_g_dist.copy()
    
    # Sort distances (smallest distance first)
    initial_rank = np.argsort(original_dist, axis=1)
    
    nn_k1 = []
    for i in range(prob.shape[0]):
        # Forward k-nearest neighbors
        forward_k1 = initial_rank[i, :k1 + 1]
        # Backward k-nearest neighbors for the candidates
        backward_k1 = initial_rank[forward_k1, :k1 + 1]
        
        # Find which candidates consider 'i' as a top neighbor (reciprocity)
        fi = np.where(backward_k1 == i)[0]
        nn_k1.append(forward_k1[fi])
        
    jaccard_dist = np.zeros_like(original_dist)
    for i in range(prob.shape[0]):
        # Only look at images within a reasonable distance threshold (0.6)
        # This speeds up calculation and removes extreme outliers
        ind_non_zero = np.where(original_dist[i, :] < 0.6)[0]
        
        # Check mutual neighborhood sets
        ind_images = [inv for inv in ind_non_zero if len(np.intersect1d(nn_k1[i], nn_k1[inv])) > 0]
        
        for j in ind_images:
            intersection = len(np.intersect1d(nn_k1[i], nn_k1[j]))
            union = len(np.union1d(nn_k1[i], nn_k1[j]))
            jaccard_dist[i, j] = 1 - (intersection / (union + 1e-12))
            
    # Combine original distance with Jaccard (re-ranking) distance
    final_dist = jaccard_dist * lambda_value + original_dist * (1 - lambda_value)
    
    # Return as a similarity matrix (1 - distance)
    return 1 - final_dist


# Training & Inference Engine

In [11]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_meter = 0
    
    for i, (imgs, labels) in enumerate(tqdm(loader, desc="Training", leave=False)):
        imgs, labels = imgs.to(Config.device), labels.to(Config.device)
        
        with torch.amp.autocast('cuda'):
            logits = model(imgs, labels)
            loss = criterion(logits, labels)
            loss = loss / Config.grad_accum
            
        scaler.scale(loss).backward()
        
        if (i + 1) % Config.grad_accum == 0:
            # --- Added: Gradient Clipping ---
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # --------------------------------
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
        loss_meter += loss.item() * Config.grad_accum
        
    return loss_meter / len(loader)

def train_epoch_sam(model, loader, optimizer, criterion, scaler):
    model.train()
    loss_meter = 0
    
    for imgs, labels in tqdm(loader, desc="Training"):
        imgs, labels = imgs.to(Config.device), labels.to(Config.device)
        
        # --- FIRST STEP (Ascent) ---
        with torch.amp.autocast('cuda'):
            logits = model(imgs, labels)
            loss = criterion(logits, labels)
        
        # Scale and backward
        optimizer.zero_grad()
        scaler.scale(loss).backward()

        # Check for NaN gradients before stepping
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        if torch.isnan(grad_norm):
            print("Skipping step: NaN detected in gradients!")
            optimizer.zero_grad()
            continue # Skip this batch
        
        # Unscale so SAM can calculate the correct gradient norm
        #scaler.unscale_(optimizer) 
        optimizer.first_step(zero_grad=True)
        
        # --- SECOND STEP (Actual Update) ---
        with torch.amp.autocast('cuda'):
            logits = model(imgs, labels)
            loss_second = criterion(logits, labels)
            
        scaler.scale(loss_second).backward()
        
        # Final step
        optimizer.second_step(zero_grad=True)
        # Note: We don't call scaler.step() here because SAM's second_step 
        # calls the base optimizer's step internally.
        scaler.update() 
        
        loss_meter += loss.item()
        
    return loss_meter / len(loader)
    
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, names = [], []
    for imgs, fnames in tqdm(loader, desc="Inference"):
        imgs = imgs.to(Config.device)
        
        # Original forward pass
        f1 = model(imgs)
        
        # Test Time Augmentation (Horizontal Flip)
        if Config.use_tta:
            f2 = model(torch.flip(imgs, [3]))
            f1 = (f1 + f2) / 2
            
        feats.append(F.normalize(f1, dim=1).cpu())
        names.extend(fnames)
    return torch.cat(feats, dim=0).numpy(), names

# Execution (Main Loop)

In [12]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import WeightedRandomSampler

# --- PATHS ---
TRAIN_CSV = "/kaggle/input/jaguar-re-id/train.csv"
TEST_CSV = "/kaggle/input/jaguar-re-id/test.csv"
TRAIN_DIR = "/kaggle/input/jaguar-re-id/train/train"
TEST_DIR = "/kaggle/input/jaguar-re-id/test/test"

# --- DATA LOADING ---
full_train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# --- SPLIT SETUP (STRATIFIED K-FOLD) ---
skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
# Create a dummy fold column
full_train_df["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(full_train_df, full_train_df["ground_truth"])):
    full_train_df.loc[val_idx, "fold"] = fold

# --- SELECT DATA FOR TRAINING ---
if Config.train_full_data:
    print("🚀 Training on FULL dataset for Submission")
    train_df = full_train_df
else:
    print(f"🔬 Training on FOLD {Config.target_fold} (Validation Mode)")
    train_df = full_train_df[full_train_df["fold"] != Config.target_fold].reset_index(drop=True)
    val_df = full_train_df[full_train_df["fold"] == Config.target_fold].reset_index(drop=True)

# Update Config with actual number of classes in training set
Config.num_classes = train_df["ground_truth"].nunique()

# 1. Encode the ground_truth strings into integers (0 to 30)
encoder = LabelEncoder()
full_train_df['label_idx'] = encoder.fit_transform(full_train_df['ground_truth'])

# 2. Calculate class counts and weights using the new integer labels
# Using .value_counts().sort_index() ensures index 0 matches class 0
class_counts = full_train_df['label_idx'].value_counts().sort_index().values
#class_counts = full_train_df.groupby('label_idx').size().sort_index().tolist()
class_weights = 1.0 / class_counts

# 3. Assign the weight to every individual image
# Now t is an integer (0-30), so class_weights[t] will work perfectly
sample_weights = [class_weights[t] for t in full_train_df['label_idx'].values]

# 4. Create the Sampler
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

train_loader = DataLoader(
    JaguarDataset(train_df, TRAIN_DIR, train_transform),
    batch_size=Config.batch_size,
    shuffle=False,
    sampler=sampler, # <--- The Magic Ingredient
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

# --- MODEL SETUP ---
# model = EVABoss(num_classes=Config.num_classes, class_counts = class_counts).to(Config.device)
# model = EVABossV2(model_name = Config.model_name, num_classes=Config.num_classes).to(Config.device)
model = EVABossDualHead(num_classes=Config.num_classes, class_counts = class_counts).to(Config.device)

# Advanced Optimizer Setup with LLRD
optimizer_params = get_optimizer_params(
    model, 
    encoder_lr=Config.encoder_lr, 
    head_lr=Config.head_lr, 
    weight_decay=Config.weight_decay
)
optimizer = torch.optim.AdamW(optimizer_params)
#optimizer = SAM(optimizer_params, torch.optim.AdamW, rho=0.05, lr = Config.encoder_lr)
scaler = torch.amp.GradScaler('cuda')
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=Config.num_epochs, eta_min=1e-6
)

#weights_tensor = torch.from_numpy(class_weights).float().to(Config.device)
#criterion = nn.CrossEntropyLoss(weight = weights_tensor, label_smoothing=0.03)
criterion = nn.CrossEntropyLoss(label_smoothing=0.03)

🚀 Training on FULL dataset for Submission


model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

In [13]:
def save_checkpoint(model, epoch, score, name="EVA-02_best_model.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'Config': {k: v for k, v in Config.__dict__.items() if not k.startswith("__")},
        'score': score
    }
    torch.save(checkpoint, name)
    print(f"✅ Model saved to {name} with score: {score:.4f}")

In [14]:
# --- EPOCH CONFIG ---
Config.num_epochs_phase_1 = 6  # Balanced ArcFace
Config.num_epochs_phase_2 = 6  # Dynamic Sub-Center ArcFace
Config.num_epochs = Config.num_epochs_phase_1 + Config.num_epochs_phase_2

print(f"🔥 Starting Dual-Phase Training | {Config.num_epochs} Total Epochs")

# --- TRAINING LOOP ---
for epoch in range(Config.num_epochs):
    
    # --- PHASE SWITCH LOGIC ---
    if epoch == Config.num_epochs_phase_1:
        print("\n🔄 PHASE SHIFT: Activating Dynamic Sub-Center ArcFace!")
        model.use_subcenter = True
        
        # 🛡️ THE WARM-START TRICK
        # Copy the learned weights from Phase 1 into all the sub-centers of Phase 2
        with torch.no_grad():
            base_weights = model.head_balanced.weight.data # Shape: (Classes, Dim)
            # Repeat the weights 3 times to fill the Sub-Center tensor
            model.head_subcenter.weight.data = base_weights.repeat_interleave(3, dim=0)
            print("✅ Sub-Centers successfully initialized with Phase 1 weights.")

    # You can keep using your existing `train_epoch` function here!
    loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
    scheduler.step()
    
    current_lr = optimizer.param_groups[0]['lr']
    phase_name = "Sub-Center" if model.use_subcenter else "Balanced"
    print(f"Epoch {epoch+1}/{Config.num_epochs} [{phase_name}] | Loss: {loss:.4f} | LR: {current_lr:.2e}")

# Usage at the end of your training loop:
save_checkpoint(model, epoch=Config.num_epochs, score=loss)

🔥 Starting Dual-Phase Training | 2 Total Epochs


Training:   0%|          | 0/473 [00:00<?, ?it/s]

Epoch 1/2 [Balanced] | Loss: 14.6593 | LR: 1.00e-06

🔄 PHASE SHIFT: Activating Dynamic Sub-Center ArcFace!
✅ Sub-Centers successfully initialized with Phase 1 weights.


Training:   0%|          | 0/473 [00:00<?, ?it/s]

Epoch 2/2 [Sub-Center] | Loss: 7.8337 | LR: 3.00e-05
✅ Model saved to EVA-02_best_model.pth with score: 7.8337


In [15]:
# # --- TRAINING LOOP ---
# print(f"🔥 Starting Training: EVA-02 Large | {Config.num_epochs} Epochs")

# for epoch in range(Config.num_epochs):
#     loss = train_epoch(model, train_loader, optimizer, criterion, scaler)
#     # loss = train_epoch_sam(model, train_loader, optimizer, criterion, scaler)
#     scheduler.step()
#     current_lr = optimizer.param_groups[0]['lr']
#     print(f"Epoch {epoch+1}/{Config.num_epochs} | Loss: {loss:.4f} | LR: {current_lr:.2e}")


# # Usage at the end of your training loop:
# save_checkpoint(model, epoch=Config.num_epochs, score=loss)

In [17]:
def load_checkpoint(filepath, num_classes, class_counts):
    # 1. Load the data from disk
    checkpoint = torch.load(filepath, map_location=Config.device)
    
    # 2. Re-initialize the exact same architecture
    # Note: Use the model class (EVABoss or MegaDBoss) you used for training
    #model = EVABoss(num_classes=num_classes, class_counts=class_counts)
    model = EVABossDualHead(num_classes=Config.num_classes, class_counts = class_counts)
    
    # 3. Load the weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(Config.device)
    model.eval() # Set to evaluation mode
    
    print(f"🚀 Loaded model from epoch {checkpoint['epoch']} (Saved score: {checkpoint['score']:.4f})")
    return model

# Usage:
model = load_checkpoint("EVA-02_best_model.pth", Config.num_classes, class_counts)

🚀 Loaded model from epoch 2 (Saved score: 7.8337)


In [18]:

# --- INFERENCE ---
print("\n🔮 Starting Inference...")
unique_test_imgs = sorted(set(test_df["query_image"]) | set(test_df["gallery_image"]))
test_loader = DataLoader(
    JaguarDataset(pd.DataFrame({"filename": unique_test_imgs}), TEST_DIR, test_transform, is_test=True),
    batch_size=Config.batch_size * 2,
    shuffle=False,
    num_workers=2
)

# Extract Features
import torch.nn.functional as F

# 1. Extract Features (Assume emb is torch.Tensor for DBA, then numpy for Rerank)
emb, names = extract_features(model, test_loader) 
img_map = {n: i for i, n in enumerate(names)}
# Note: Ensure emb is on GPU for fast DBA
emb_tensor = torch.from_numpy(emb).to(Config.device)

# --- STEP 1: DATABASE AUGMENTATION (DBA) ---
# We apply DBA to every image in the set first. 
# This "denoises" the features of every jaguar by looking at its neighbors.
if Config.use_dba:
    print("Applying DBA...")
    emb_tensor = apply_dba(emb_tensor, k=3)

# --- STEP 2: QUERY EXPANSION (QE) ---
# Now we enhance the queries using the already-cleaned DBA features.
if Config.use_qe:
    # Using the weighted version helps Bernard (rare classes)
    emb_tensor = query_expansion(emb_tensor, top_k=3)

# Move back to CPU/Numpy for the Re-ranking step
emb_final = emb_tensor#.cpu().numpy()


# --- STEP 3: SIMILARITY & RE-RANKING ---
sim_matrix = emb_final @ emb_final.T

if Config.use_rerank:
    # k-reciprocal usually works better on the "refined" sim_matrix
    sim_matrix = k_reciprocal_rerank(sim_matrix, k1=20, k2=6, lambda_value=0.3)


🔮 Starting Inference...


Inference:   0%|          | 0/47 [00:00<?, ?it/s]

Applying DBA...
Applying Weighted Query Expansion (k=3)...
Applying K-Reciprocal Re-ranking...


In [19]:
# --- GENERATE SUBMISSION ---

preds = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Mapping Predictions"):
    idx_q = img_map[row["query_image"]]
    idx_g = img_map[row["gallery_image"]]
    
    score = sim_matrix[idx_q, idx_g]
    preds.append(max(0.0, min(1.0, score))) # Clip to valid range

sub = pd.DataFrame({"row_id": test_df["row_id"], "similarity": preds})
sub.to_csv("submission.csv", index=False)

print(f"✅ Submission Saved! Mean Similarity: {np.mean(preds):.4f}")

Mapping Predictions:   0%|          | 0/137270 [00:00<?, ?it/s]

✅ Submission Saved! Mean Similarity: 0.3256
